# Welsh ASR — XLS-R fine-tuning on ColabRun this on a **T4 GPU** runtime (Runtime -> Change runtime type -> T4 GPU).Only the training loop runs here. The baseline, preprocessing, error analysisand the demo all run locally on CPU — GPU minutes are the scarce resource.**Secrets:** put `HF_TOKEN`, `WANDB_API_KEY` and `GH_TOKEN` in the ColabSecrets panel (key icon in the left sidebar) and enable them for this notebook.Never paste a token into a cell — this notebook is shareable.

## 1. Confirm a GPU was actually allocated

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv# Free tier sometimes hands you a CPU-only runtime. If this errors, stop and# retry later rather than training on CPU.

## 2. Install dependencies

In [ ]:
!pip install -q "transformers>=5.0" "datasets>=5.0" jiwer accelerate soundfile librosa wandb

## 3. Get the codeThe repo is private, so cloning needs a GitHub token with `repo` scope inColab Secrets as `GH_TOKEN`. If you have made the repo public, the fallbackclone works without one.

In [ ]:
import os, subprocessfrom google.colab import userdataREPO = "paarthN/welsh-asr"try:    tok = userdata.get("GH_TOKEN")    url = f"https://{tok}@github.com/{REPO}.git"except Exception:    url = f"https://github.com/{REPO}.git"if not os.path.exists("/content/welsh-asr"):    subprocess.run(["git", "clone", url, "/content/welsh-asr"], check=True)%cd /content/welsh-asr!git log --oneline -1

## 4. Secrets and W&B

In [ ]:
from google.colab import userdataimport osos.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")os.environ["WANDB_PROJECT"] = "welsh-asr"import wandb; wandb.login()

## 5. Mount Drive for checkpointsThis is what makes a disconnect survivable. Each checkpoint is ~3.6GB(model + optimizer state), and `save_total_limit=2` keeps two, so budget ~7GBof the free 15GB.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")CKPT_DIR = "/content/drive/MyDrive/welsh-asr-xlsr"!mkdir -p "{CKPT_DIR}"!ls -la "{CKPT_DIR}"

## 6. Build the vocabulary and processor (CPU, seconds)

In [ ]:
!cd src && python prepare_data.py

## 7. TrainStart at the top of the ladder and step down **only** if you hit CUDA OOM.Effective batch stays 16 throughout, so the learning dynamics do not shift.| Try | `--batch-size` | `--grad-accum` | `--gradient-checkpointing` ||---|---|---|---|| 1 | 4 | 4 | off || 2 | 2 | 8 | off || 3 | 2 | 8 | on || 4 | 2 | 8 | on, plus `--max-duration 20` |`max_steps=2100` is roughly 10 epochs over the ~2.8k usable clips. The targetis simply to beat the zero-shot Whisper-small baseline; do not grind epochschasing decimal places.

In [ ]:
!cd src && python -u train.py \    --output-dir "{CKPT_DIR}" \    --max-steps 2100 \    --batch-size 4 \    --grad-accum 4 \    --hub-model-id pnawani/welsh-asr-xlsr-300m \    --push-to-hub

## 8. Resume after a disconnectRe-run cells 1-6, then this instead of cell 7. It picks up from the lastcheckpoint on Drive.

In [ ]:
!cd src && python -u train.py \    --output-dir "{CKPT_DIR}" \    --max-steps 2100 \    --batch-size 4 \    --grad-accum 4 \    --hub-model-id pnawani/welsh-asr-xlsr-300m \    --push-to-hub \    --resume

## 9. Evaluate the fine-tuned model on the test setWrites `results/finetuned_results.json`. Download it and run`src/error_analysis.py` locally against it plus the Whisper baseline.

In [ ]:
!cd src && python -u evaluate.py \    --model "{CKPT_DIR}" \    --out ../results/finetuned_results.json \    --progress-every 50

In [ ]:
from google.colab import filesfiles.download("/content/welsh-asr/results/finetuned_results.json")